# **SAS Evaluation**
This notebook measures *Confirmation Bias* using the `stsb-roberta-large` model to weigh the semantic similarity between sentences.

In [ ]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.sas import compute_sas_metrics

## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [ ]:
# Define the path to the generated results (JSONL format)
file_path = "../data/interim/3_fever_deepseek_r1_results.jsonl"
# file_path = "../data/interim/4_truthfulqa_deepseek_r1_results.jsonl"
# file_path = "../data/interim/5_mmlu_pro_deepseek_r1_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
    display(df_results.head(1))
except FileNotFoundError:
    print(f"File not found at the requested path ({file_path}).")

## **Evaluation**
Compute the evaluation metrics by sending the generated results to the model, that calculates the Separation margin and the CB_SAS score.

In [ ]:
# Compute the SAS metrics
df_evaluated = compute_sas_metrics(df_results, tau_sep=0.03)

## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [ ]:
# Aggregation of the overall Confirmation Bias mean
mean_cb = float(df_evaluated["CB_SAS"].mean())

# Aggregation based on the Reliability Gate generated (Sep >= 0.03)
reliable_mean_cb = float(df_evaluated[df_evaluated["sas_reliable"] == True]["CB_SAS"].mean())
reliable_rate = float(df_evaluated["sas_reliable"].mean())

print("Results of the SAS Evaluation:")
print(f"Mean Confirmation Bias (All samples): {mean_cb:.6f}")
print(f"Mean Confirmation Bias (Reliable samples): {reliable_mean_cb:.6f}")
print(f"Reliability Rate (% of Sep >= TAU):  {reliable_rate:.2%}")

display(df_evaluated[["sample", "model", "Sep", "CB_SAS", "CB_SAS_clipped", "sas_reliable"]].head())